In [9]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import os
import torch.nn.functional as F
from tqdm import tqdm
from lib.networks_prototype import Prototype_CASCADE
from utils.dataloader import test_dataset

def analyze_model_statistics(cfg):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    num_fg = cfg['num_fg']
    num_bg = cfg['num_bg']
    model_path = cfg['model_path']
    
    print(f"Initializing model with FG={num_fg}, BG={num_bg}...")
    model = Prototype_CASCADE(num_fg=num_fg, num_bg=num_bg).to(device)
    
    if os.path.exists(model_path):
        print(f"Loading weights from: {model_path}")
        state_dict = torch.load(model_path, map_location=device, weights_only=True)
        model.load_state_dict(state_dict, strict=False)
    else:
        print(f"Error: Model path not found at {model_path}")
        return
    model.eval()

    image_root = cfg['val_image_dir']
    gt_root = cfg['val_mask_dir']
    test_loader = test_dataset(image_root, gt_root, testsize=352)

    fg_pixel_counts = np.zeros(num_fg)
    bg_pixel_counts = np.zeros(num_bg)
    
    print("Running inference...")
    with torch.no_grad():
        for i in tqdm(range(test_loader.size)):
            image, gt, name = test_loader.load_data()
            
            image = image.cuda()
            
            gt = np.asarray(gt, np.float32)
            gt /= (gt.max() + 1e-8)
            mask = torch.from_numpy(gt).cuda().unsqueeze(0).unsqueeze(0)
            
            _, similarity_map_high = model(image)
            
            if similarity_map_high.shape[-2:] != mask.shape[-2:]:
                similarity_map_high = F.interpolate(similarity_map_high, size=mask.shape[-2:], 
                                                    mode='bilinear', align_corners=False)

            # --- Foreground Analysis ---
            fg_sim = similarity_map_high[:, :num_fg, :, :]
            fg_assignments = torch.argmax(fg_sim, dim=1)
            valid_fg_pixels = fg_assignments[mask.squeeze(1) > 0.5]
            
            if valid_fg_pixels.numel() > 0:
                counts_fg = torch.bincount(valid_fg_pixels, minlength=num_fg)
                fg_pixel_counts += counts_fg.cpu().numpy()

            # --- Background Analysis ---
            bg_sim = similarity_map_high[:, num_fg:, :, :]
            bg_assignments = torch.argmax(bg_sim, dim=1)
            valid_bg_pixels = bg_assignments[mask.squeeze(1) <= 0.5]

            if valid_bg_pixels.numel() > 0:
                counts_bg = torch.bincount(valid_bg_pixels, minlength=num_bg)
                bg_pixel_counts += counts_bg.cpu().numpy()

    # --- Plotting ---
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Plot 1: Foreground
    total_fg = fg_pixel_counts.sum()
    if total_fg > 0:
        fg_ratios = fg_pixel_counts / total_fg
        x_fg = np.arange(num_fg) + 1
        bars_fg = axes[0].bar(x_fg, fg_ratios, color='skyblue', edgecolor='black')
        axes[0].set_xlabel('Foreground Prototype ID')
        axes[0].set_ylabel('Usage Ratio')
        axes[0].set_title(f'Foreground Utilization (K={num_fg})')
        axes[0].set_ylim(0, 1.05)
        axes[0].set_xticks(x_fg)
        
        for bar in bars_fg:
            yval = bar.get_height()
            axes[0].text(bar.get_x() + bar.get_width()/2, yval + 0.01, f'{yval:.2f}', ha='center', va='bottom')
    else:
        axes[0].text(0.5, 0.5, 'No FG Pixels', ha='center', va='center')

    # Plot 2: Background
    total_bg = bg_pixel_counts.sum()
    if total_bg > 0:
        bg_ratios = bg_pixel_counts / total_bg
        x_bg = np.arange(num_bg) + 1
        bars_bg = axes[1].bar(x_bg, bg_ratios, color='lightcoral', edgecolor='black')
        axes[1].set_xlabel('Background Prototype ID')
        axes[1].set_ylabel('Usage Ratio')
        axes[1].set_title(f'Background Utilization (K={num_bg})')
        axes[1].set_ylim(0, 1.05)
        axes[1].set_xticks(x_bg)
        
        for bar in bars_bg:
            yval = bar.get_height()
            axes[1].text(bar.get_x() + bar.get_width()/2, yval + 0.01, f'{yval:.2f}', ha='center', va='bottom')
    else:
        axes[1].text(0.5, 0.5, 'No BG Pixels', ha='center', va='center')

    # Final Layout
    plt.suptitle(cfg.get('plot_title', 'Prototype Utilization Analysis'), fontsize=16)
    plt.tight_layout()
    
    save_name = cfg.get('save_name', 'utilization_full_plot.png')
    plt.savefig(save_name)
    plt.close()
    print(f"Saved combined plot to {save_name}")

if __name__ == '__main__':
    config = {
        'num_fg': 4,
        'num_bg': 4,
        'model_path': '/home/U116med/wch_code/cascade/models/prototype_cascade/prototype_kl_loss_0.05_fb_44/best.pth',
        'val_image_dir': '/home/U116med/data/polyp/TestDataset/test/images/',
        'val_mask_dir': '/home/U116med/data/polyp/TestDataset/test/masks/',
        'plot_title': 'Prototype Utilization Analysis (K=4+4) KL Loss 0.05',
        'save_name': 'utilization_fg4_bg4_kl_0.05.png'
    }

    analyze_model_statistics(config)

Initializing model with FG=4, BG=4...
Model decoder:  created, param count: 1913515
Loading weights from: /home/U116med/wch_code/cascade/models/prototype_cascade/prototype_kl_loss_0.05_fb_44/best.pth
Running inference...


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 798/798 [00:22<00:00, 34.89it/s]


Saved combined plot to utilization_fg4_bg4_kl_0.15.png
